# 07 — ClaimIQ Controlled Streaming Simulation

**Week:** 10  
**Project:** ClaimIQ – Insurance Risk Analytics  
**Purpose:** Process the six approved ClaimIQ `claim_status_event.json` drops incrementally with an explicit v1.0 schema, 2-hour watermark, `event_id` deduplication, quarantine, controlled schema drift retention, and a stable checkpoint.

> **Important:** This notebook is prepared for ClaimIQ, not a generic streaming example. It replaces the starter notebook's generic `entity_id / severity` event model with the ClaimIQ claim-status event contract.


## Week 10 contract

The six controlled drops are:

| Drop | Rows | Focus |
|---|---:|---|
| `drop_01_baseline` | 20 | valid submissions |
| `drop_02_progression` | 20 | under-review progression / sequence |
| `drop_03_duplicate_late_sequence` | 18 | duplicate, late and out-of-order behavior |
| `drop_04_malformed_missing_orphan` | 15 | malformed, missing-required and orphan behavior |
| `drop_05_monetary_and_state` | 16 | monetary and lifecycle-state failures |
| `drop_06_schema_drift` | 16 | controlled v1.1 extra fields |

The implementation must retain rescued/schema-drift data, deduplicate `event_id`, apply a 2-hour watermark on `event_timestamp`, quarantine invalid events, write accepted events to `fact_claim_event_stream`, and support a no-new-file rerun with zero new physical records.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    LongType, DecimalType, BooleanType
)

# ClaimIQ Week 10 paths
# If your team placed the six drops somewhere else, change ONLY STREAM_INPUT_ROOT.
STREAM_INPUT_ROOT = "/Volumes/workspace/default/claimiq/data_sample/streaming/"
CHECKPOINT_PATH = "/Volumes/workspace/default/claimiq/streaming/checkpoints/claim_status"

ACCEPTED_TABLE = "fact_claim_event_stream"
QUARANTINE_TABLE = "quarantine_claim_event_stream"
LOG_TABLE = "streaming_claim_status_log"

print("Input root:", STREAM_INPUT_ROOT)
print("Checkpoint:", CHECKPOINT_PATH)


## Step 1 — Explicit ClaimIQ v1.0 event schema

The schema below keeps the ClaimIQ business fields typed. The `_rescued_data` column is retained separately by Auto Loader so Drop 06's v1.1 extra fields are not silently discarded.


In [ ]:
event_schema_v1 = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("event_sequence", LongType(), True),

    StructField("claim_id", StringType(), True),
    StructField("policy_id", StringType(), True),
    StructField("policyholder_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("provider_id", StringType(), True),
    StructField("claim_type", StringType(), True),
    StructField("claim_status", StringType(), True),

    StructField("requested_amount", DecimalType(18, 2), True),
    StructField("approved_amount", DecimalType(18, 2), True),
    StructField("paid_amount", DecimalType(18, 2), True),
    StructField("reserve_amount", DecimalType(18, 2), True),

    StructField("risk_band", StringType(), True),
    StructField("review_flag", BooleanType(), True),
    StructField("currency_code", StringType(), True),
])

print("ClaimIQ v1.0 schema prepared.")


## Step 2 — Read only the approved `claim_status_event.json` files

The exact filename is required by the ClaimIQ Week 10 contract. Auto Loader is used for incremental file discovery, while schema rescue preserves fields that are present in Drop 06 but are not part of v1.0.


In [ ]:
raw_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .schema(event_schema_v1)
        .load(STREAM_INPUT_ROOT)
)

stream_df = (
    raw_stream
        .withColumn("_source_file", F.input_file_name())
        .filter(F.element_at(F.split(F.col("_source_file"), "/"), -1) == "claim_status_event.json")
        .withColumn(
            "drop_name",
            F.regexp_extract(F.col("_source_file"), r"(drop_\d+_[^/]+)", 1)
        )
)

display(stream_df.select(
    "drop_name", "event_id", "event_sequence", "claim_id",
    "claim_status", "event_timestamp", "_rescued_data"
))


## Step 3 — Watermark and `event_id` deduplication

The watermark is applied to `event_timestamp` before deduplication, as required for the Week 10 proof. A late event is therefore handled according to the two-hour policy instead of being silently treated as a normal duplicate.


In [ ]:
deduped_stream = (
    stream_df
        .withWatermark("event_timestamp", "2 hours")
        .dropDuplicates(["event_id"])
)

display(deduped_stream.select(
    "drop_name", "event_id", "event_sequence",
    "claim_id", "claim_status", "event_timestamp"
))


## Step 4 — Validate each micro-batch and route Accepted vs Quarantine

Validation is performed at the ClaimIQ event level.

Checks covered:
- required identifiers / event fields
- policyholder, policy, product and provider reference resolution
- non-negative monetary values and requested/approved/paid limits
- event sequence progression
- non-regressing lifecycle status
- rescued/schema-drift visibility

The accepted sink is the Week 10 live event fact. Invalid records are retained with a reason instead of disappearing.


In [ ]:
from pyspark.sql.window import Window

STATUS_RANK = (
    F.when(F.lower(F.col("claim_status")) == "submitted", F.lit(1))
     .when(F.lower(F.col("claim_status")) == "under_review", F.lit(2))
     .when(F.lower(F.col("claim_status")).isin("approved", "denied"), F.lit(3))
     .when(F.lower(F.col("claim_status")) == "paid", F.lit(4))
     .when(F.lower(F.col("claim_status")) == "closed", F.lit(5))
     .otherwise(F.lit(None).cast("int"))
)

def process_claim_event_batch(batch_df, batch_id):
    if batch_df.limit(1).count() == 0:
        return

    b = (
        batch_df
        .withColumn("status_rank", STATUS_RANK)
        .withColumn("_processed_batch_id", F.lit(int(batch_id)))
        .withColumn("_processed_at", F.current_timestamp())
    )

    # Reference data: use the already-approved Trusted Silver layer.
    policies = spark.table("trusted_silver_claimiq_policies").select(
        "policy_id", "policyholder_id", "product_id", "coverage_limit"
    ).dropDuplicates(["policy_id"])

    policyholders = spark.table("trusted_silver_claimiq_policyholders").select(
        "policyholder_id"
    ).dropDuplicates(["policyholder_id"])

    products = spark.table("trusted_silver_claimiq_products").select(
        "product_id"
    ).dropDuplicates(["product_id"])

    providers = spark.table("trusted_silver_claimiq_providers").select(
        "provider_id"
    ).dropDuplicates(["provider_id"])

    # Existing accepted events are used only for state/sequence comparison.
    if spark.catalog.tableExists(ACCEPTED_TABLE):
        prior = spark.table(ACCEPTED_TABLE).select(
            "claim_id", "event_sequence", "claim_status", "event_timestamp"
        )
    else:
        prior = spark.createDataFrame(
            [],
            "claim_id string, event_sequence long, claim_status string, event_timestamp timestamp"
        )

    prior_latest = (
        prior
        .withColumn("prior_status_rank", (
            F.when(F.lower(F.col("claim_status")) == "submitted", F.lit(1))
             .when(F.lower(F.col("claim_status")) == "under_review", F.lit(2))
             .when(F.lower(F.col("claim_status")).isin("approved", "denied"), F.lit(3))
             .when(F.lower(F.col("claim_status")) == "paid", F.lit(4))
             .when(F.lower(F.col("claim_status")) == "closed", F.lit(5))
             .otherwise(F.lit(None).cast("int"))
        ))
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("claim_id").orderBy(
                    F.col("event_sequence").desc_nulls_last(),
                    F.col("event_timestamp").desc_nulls_last()
                )
            )
        )
        .filter("rn = 1")
        .drop("rn")
    )

    b = (
        b
        .join(
            policies.withColumnRenamed("policyholder_id", "_policyholder_ref")
                    .withColumnRenamed("product_id", "_product_ref")
                    .withColumnRenamed("coverage_limit", "_coverage_limit"),
            "policy_id", "left"
        )
        .join(
            policyholders.withColumnRenamed("policyholder_id", "_ph_ref"),
            b.policyholder_id == F.col("_ph_ref"), "left"
        )
        .join(
            products.withColumnRenamed("product_id", "_product_ref2"),
            b.product_id == F.col("_product_ref2"), "left"
        )
        .join(
            providers.withColumnRenamed("provider_id", "_provider_ref"),
            b.provider_id == F.col("_provider_ref"), "left"
        )
        .join(
            prior_latest.withColumnRenamed("claim_id", "_prior_claim_id")
                        .withColumnRenamed("event_sequence", "_prior_sequence")
                        .withColumnRenamed("claim_status", "_prior_status")
                        .withColumnRenamed("prior_status_rank", "_prior_status_rank"),
            b.claim_id == F.col("_prior_claim_id"), "left"
        )
    )

    b = b.withColumn(
        "dq_reason",
        F.when(F.col("event_id").isNull(), "missing_event_id")
         .when(F.col("event_timestamp").isNull(), "missing_event_timestamp")
         .when(F.col("claim_id").isNull(), "missing_claim_id")
         .when(F.col("policy_id").isNull(), "missing_policy_id")
         .when(F.col("event_sequence").isNull(), "missing_event_sequence")
         .when(F.col("_ph_ref").isNull(), "orphan_policyholder")
         .when(F.col("_policy_ref").isNull(), "orphan_policy")
         .when(F.col("_product_ref2").isNull(), "orphan_product")
         .when(F.col("provider_id").isNotNull() & F.col("_provider_ref").isNull(), "orphan_provider")
         .when(F.coalesce(F.col("requested_amount"), F.lit(0)) < 0, "negative_requested_amount")
         .when(F.coalesce(F.col("approved_amount"), F.lit(0)) < 0, "negative_approved_amount")
         .when(F.coalesce(F.col("paid_amount"), F.lit(0)) < 0, "negative_paid_amount")
         .when(F.coalesce(F.col("reserve_amount"), F.lit(0)) < 0, "negative_reserve_amount")
         .when(F.col("approved_amount") > F.col("requested_amount"), "approved_exceeds_requested")
         .when(F.col("paid_amount") > F.col("requested_amount"), "paid_exceeds_requested")
         .when(F.col("_coverage_limit").isNotNull() & (F.col("paid_amount") > F.col("_coverage_limit")), "paid_exceeds_coverage_limit")
         .when(F.col("_prior_sequence").isNotNull() & (F.col("event_sequence") <= F.col("_prior_sequence")), "non_increasing_event_sequence")
         .when(
             F.col("_prior_status_rank").isNotNull() &
             F.col("status_rank").isNotNull() &
             (F.col("status_rank") < F.col("_prior_status_rank")),
             "non_regressing_status"
         )
         .when(F.col("_rescued_data").isNotNull(), "schema_drift_review")
    )

    # Schema drift is retained, not rejected. It is accepted with rescued context.
    b = b.withColumn(
        "route",
        F.when(F.col("dq_reason").isNull() | (F.col("dq_reason") == "schema_drift_review"), "accepted")
         .otherwise("quarantine")
    )

    accepted = (
        b.filter(F.col("route") == "accepted")
         .drop(
             "_policyholder_ref", "_policy_ref", "_product_ref", "_product_ref2",
             "_ph_ref", "_provider_ref", "_coverage_limit",
             "_prior_claim_id", "_prior_sequence", "_prior_status",
             "_prior_status_rank", "status_rank", "dq_reason", "route"
         )
    )

    quarantine = (
        b.filter(F.col("route") == "quarantine")
         .withColumn("quarantine_reason", F.col("dq_reason"))
         .withColumn("raw_event_json", F.to_json(F.struct(*[F.col(c) for c in batch_df.columns])))
         .select(
             "event_id", "event_timestamp", "event_sequence", "claim_id",
             "policy_id", "policyholder_id", "product_id", "provider_id",
             "claim_status", "quarantine_reason", "raw_event_json",
             "_rescued_data", "drop_name", "_source_file",
             "_processed_batch_id", "_processed_at"
         )
    )

    accepted.write.format("delta").mode("append").saveAsTable(ACCEPTED_TABLE)
    quarantine.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)

    log_df = (
        b.groupBy("drop_name", "route")
         .count()
         .withColumn("batch_id", F.lit(int(batch_id)))
         .withColumn("logged_at", F.current_timestamp())
    )
    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

    print(f"batch_id={batch_id} processed")


## Step 5 — Start the ClaimIQ stream

`availableNow=True` is intentional for the controlled six-drop simulation: the run processes all currently available files, updates the stable checkpoint, and stops. Add the next controlled drop and rerun the same notebook to demonstrate incremental processing.


In [ ]:
query = (
    deduped_stream
        .writeStream
        .foreachBatch(process_claim_event_batch)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .outputMode("append")
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()
print("ClaimIQ Week 10 streaming run completed.")


## Step 6 — Drop-by-drop reconciliation

Use this after each controlled run. The physical input rows must reconcile to Accepted + Quarantine for the processed drop. Do not use only `event_id` for reconciliation; the Week 10 contract requires drop-level physical-line evidence.


In [ ]:
display(
    spark.table(LOG_TABLE)
        .groupBy("drop_name", "route")
        .agg(F.sum("count").alias("rows"))
        .orderBy("drop_name", "route")
)

display(
    spark.table(ACCEPTED_TABLE)
        .groupBy("drop_name")
        .count()
        .orderBy("drop_name")
)

display(
    spark.table(QUARANTINE_TABLE)
        .groupBy("drop_name", "quarantine_reason")
        .count()
        .orderBy("drop_name", "quarantine_reason")
)


## Step 7 — Required duplicate / late / schema-drift proof

These checks are evidence checks, not dashboard decoration.

1. Duplicate `event_id` must not create two accepted records.
2. Drop 03 late/out-of-order behavior must be visible in the accepted/quarantine evidence.
3. Drop 06 extra fields must remain visible through `_rescued_data`.
4. Restarting with no new files must add zero accepted/quarantined rows.


In [ ]:
# Duplicate event_id proof
display(
    spark.table(ACCEPTED_TABLE)
        .groupBy("event_id")
        .count()
        .filter("count > 1")
)

# Rescued/schema-drift proof
display(
    spark.table(ACCEPTED_TABLE)
        .filter("_rescued_data IS NOT NULL")
        .select("drop_name", "event_id", "_rescued_data")
        .orderBy("drop_name", "event_id")
)

# Quarantine proof
display(
    spark.table(QUARANTINE_TABLE)
        .groupBy("quarantine_reason")
        .count()
        .orderBy(F.desc("count"))
)


## Step 8 — No-new-file rerun proof

Run the streaming cell again without adding any new input file.

**Pass condition:** accepted-row count and quarantine-row count remain unchanged.

This is the Week 10 idempotency/checkpoint proof. Do not delete or reset the checkpoint for this test.


In [ ]:
before_accepted = spark.table(ACCEPTED_TABLE).count()
before_quarantine = spark.table(QUARANTINE_TABLE).count()

print("Before rerun — accepted:", before_accepted)
print("Before rerun — quarantine:", before_quarantine)

# Re-run the Step 5 streaming cell above with the SAME checkpoint and NO new files.
# Then execute this cell again.

after_accepted = spark.table(ACCEPTED_TABLE).count()
after_quarantine = spark.table(QUARANTINE_TABLE).count()

print("After rerun — accepted:", after_accepted)
print("After rerun — quarantine:", after_quarantine)

assert after_accepted == before_accepted, "FAIL: accepted rows increased on a no-new-file rerun."
assert after_quarantine == before_quarantine, "FAIL: quarantine rows increased on a no-new-file rerun."

print("PASS: no-new-file rerun added zero accepted/quarantined physical records.")


## Step 9 — ClaimIQ Week 10 completion evidence

Before committing, capture only useful evidence:

- six-drop incremental processing log
- checkpoint path
- duplicate/late-event proof
- quarantine reasons
- Drop 06 rescued/schema-drift proof
- no-new-file rerun with zero new rows

### Team ownership

| Student | Deep ownership | Backup |
|---|---|---|
| V.Abhigna | stream schema / checkpoint | S.Vaishnavi |
| S.Vaishnavi | watermark / deduplication | Gayathri |
| Gayathri | DQ / schema drift / live output | V.Abhigna |

### AI transparency

Record:
- assistance received
- what code/wording was changed
- what was verified by the team in Databricks
- what each student can explain independently
- any unresolved limitation


## Week 10 boundary

This notebook implements the controlled streaming simulation only. Do not redesign the existing Gold model or add raw/Bronze/Silver sources to Power BI during Week 10.

The live event output is the Week 10 streaming artifact and should be reconciled before it is used to refresh the appropriate dashboard/live-simulation evidence.
